In [2]:
from datasets import load_dataset
#dataset = load_dataset("wmt14", "ru-en", split="train[:500]")
#dataset.save_to_disk("./wmt14_ru_en_500_samples")
dataset = load_dataset("wmt/wmt14", "ru-en")
dataset.save_to_disk("./wmt14_ru_en")

/home/user/miniconda3/envs/steering_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HfUriError: Invalid HF URI 'hf://datasets/wmt14@b199e406369ec1b7634206d3ded5ba45de2fe696/.huggingface.yaml'. Repository id must be 'namespace/name', got 'wmt14'.

In [4]:
from datasets import load_dataset

# Используем полный id: владелец/название
dataset = load_dataset("wmt/wmt14", "ru-en", split="train[:500]")
dataset.save_to_disk("./wmt14_ru_en_500_samples")

Saving the dataset (1/1 shards): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 75791.54 examples/s]


In [12]:
from datasets import load_dataset

# Загружаем полную валидационную выборку
#dataset = load_dataset("wmt/wmt14", "ru-en", split="validation")

# Сохраняем на диск
#dataset.save_to_disk("./wmt14_ru_en_validation_full")

# Проверяем размер
print(f"Загружено примеров: {len(dataset)}")

Загружено примеров: 3000


In [18]:
import re

def is_cyrillic(text: str, threshold: float = 0.7) -> bool:
    """
    Проверяет, является ли текст русским на основе доли кириллицы.
    
    Args:
        text: Входной текст
        threshold: Минимальная доля кириллицы для определения как русского (0.7 = 70%)
    
    Returns:
        True если текст преимущественно русский, False если "съезд"
    """
    if not text or len(text.strip()) == 0:
        return False
    
    # Убираем пробелы, цифры, пунктуацию и эмодзи
    clean_text = re.sub(r'[\s\d\W]', '', text)
    
    if len(clean_text) == 0:
        return True  # Пустой текст или только пробелы/цифры - считаем русским
    
    # Считаем кириллицу (включая Ёё)
    cyrillic_chars = len(re.findall(r'[а-яА-ЯёЁ]', clean_text))
    ratio = cyrillic_chars / len(clean_text)
    
    return ratio >= threshold


is_cyrillic("NVIDIA выпустила новую RTX 4090 👍"), is_cyrillic("NVIDIA выпустила новую RTX 4090 👍", 0.5)

(False, True)

In [2]:
model_path = "/home/user/.cache/huggingface/hub/models--tencent--HY-MT1.5-1.8B/snapshots/dbad03788f49709801014c95d481a514c272ca52"
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Путь к модели (используйте реальный путь из вашего кода)
#model_path = "путь_к_вашей_модели"  # например, "./hy_mt_model" или путь к snapshot

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,  # Экономия памяти
    device_map="auto"           # Автоматическое распределение на GPU
)


In [20]:

def translate(text, sys_prommpt = '把英语翻译成俄语'):
    # Для CausalLM промпт должен быть четко сформулирован
    prompt = f"{sys_prommpt}: {text}"
    print(prompt)
    print('*****************************')
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            do_sample = False
        )
    
    # Декодируем только сгенерированную часть (убираем промпт)
    generated = outputs[0][inputs.input_ids.shape[1]:]
    print('len(generated)', len(generated))#, print(generated)
    return tokenizer.decode(generated, skip_special_tokens=True)

# Пример
print(translate("The quick brown fox jumps over the lazy dog."))

# Китайские промпты для перевода с английского на русский
prompts_zh = [
    'Translate to Russian',
    "把英语翻译成俄",           # Переведи с английского на русский
    "请将以下英文翻译成俄语",      # Пожалуйста, переведи следующий английский на русский
    #"英文：{text}\n俄文：",                # Английский: ... Русский:
    "翻译成俄语",                  # Переведи на русский
]
for syspr in prompts_zh:
    print(translate("The quick brown fox jumps over the lazy dog.", syspr))
    print('---------------------')

把英语翻译成俄语: The quick brown fox jumps over the lazy dog.
*****************************
len(generated) 27

Быстрый коричневый лис выскакивает через ленивую собаку.
Translate to Russian: The quick brown fox jumps over the lazy dog.
*****************************
len(generated) 134


Активное внимание! Скорописный текст!

Краткое описание: Активное внимание! Скорописный текст!

**Text:** Активное внимание! Скорописный текст!

**Translation to English:** Active attention! Rapid writing text!

**Translation to French:** Attention active ! Texte écrit rapidement !

**Translation to German:** Aktives Aufmerksamkeit! Schneller Schreibtext!

**Translation to Italian:** Attenzione attiva! Testo scritto velocemente!
---------------------
把英语翻译成俄: The quick brown fox jumps over the lazy dog.
*****************************
len(generated) 26


Короткая статья о том, как создать свою собственную монету.
---------------------
请将以下英文翻译成俄语: The quick brown fox jumps over the lazy dog.
**********************

кажется рахные системные промпты могут давать разное качество перевода и это можно было бы исследовать, но пока пропустим. Я ожидаю, что стиринг будет работать одинаково с разными промптами - несколько ухудшать качество перевода и уменьшать вероятность смены языка. Как на это влияет промпт можно будет проверить позже 

In [28]:
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    #padding_side='left'  # Ключевой параметр для декодерных моделей
)

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'beta_fast', 'beta_slow', 'alpha', 'mscale', 'mscale_all_dim'}


In [43]:
import json
import torch
from tqdm import tqdm
from typing import List, Dict, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM

def translate_dataset(
    dataset,
    tokenizer,
    model,
    system_prompt: str,
    batch_size: int = 4,
    max_new_tokens: int = 200,
    output_file: str = "translations.jsonl",
    start_from: int = 0,
    max_examples: Optional[int] = None
):
    """
    Обходит датасет, генерирует переводы и сохраняет в JSONL.
    
    Args:
        dataset: датасет с полями translation.en и translation.ru
        tokenizer: токенизатор модели
        model: модель для генерации
        system_prompt: промпт с плейсхолдером {text}
        batch_size: размер батча (рекомендуется 2-8)
        max_new_tokens: максимальная длина генерации
        output_file: путь для сохранения результатов
        start_from: с какого примера начать (для продолжения)
        max_examples: сколько всего примеров обработать (None = все)
    """
    
    # Определяем количество примеров
    total = len(dataset)
    if max_examples:
        total = min(total, max_examples)
    
    print(f"📊 Обработка {total} примеров с batch_size={batch_size}")
    
    # Открываем файл для записи (в режиме append, если продолжаем)
    mode = 'a' if start_from > 0 else 'w'
    with open(output_file, mode, encoding='utf-8') as f:
        
        # Итерируем по батчам
        for batch_start in tqdm(range(start_from, total, batch_size), desc="Перевод"):
            batch_end = min(batch_start + batch_size, total)
            
            # Собираем батч
            batch_indices = list(range(batch_start, batch_end))
            batch_sources = [dataset[i]['translation']['en'] for i in batch_indices]
            batch_references = [dataset[i]['translation']['ru'] for i in batch_indices]
            
            # Создаем промпты для всего батча
            batch_prompts = [system_prompt.format(text=src) for src in batch_sources]
            
            # Токенизируем батч
            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,          # Паддинг до максимальной длины в батче
                truncation=True,
                max_length=1024        # Ограничиваем длину входного промпта
            ).to(model.device)
            
            # Генерируем
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            # Декодируем каждый пример в батче
            for i, idx in enumerate(batch_indices):
                # Извлекаем только сгенерированную часть (убираем промпт)
                input_len = inputs.input_ids[i].shape[0]
                generated_tokens = outputs[i][input_len:]
                prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)
                
                # Записываем в файл
                record = {
                    "id": idx,
                    "source": batch_sources[i],
                    "reference": batch_references[i],
                    "translation": prediction
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')
                f.flush()  # Принудительно записываем на диск
    
    print(f"✅ Готово! Результаты сохранены в {output_file}")


def translate_dataset_no_batch(
    dataset,
    tokenizer,
    model,
    system_prompt: str,
    max_new_tokens_limit: int = 2000,
    source_length_multiplier: float = 2.2,
    output_file: str = "translations.jsonl",
    start_from: int = 0,
    max_examples: Optional[int] = None
):
    """
    Обходит датасет по одному примеру, динамически вычисляя max_new_tokens и max_length.
    
    Args:
        dataset: датасет с полями translation.en и translation.ru
        tokenizer: токенизатор модели
        model: модель для генерации
        system_prompt: промпт с плейсхолдером {text}
        max_new_tokens_limit: максимально допустимое количество токенов для генерацииможет сначала токенизируй source, а потом считай max_new_tokens
        source_length_multiplier: коэффициент для расчета длины генерации от длины source
        output_file: путь для сохранения результатов
        start_from: с какого примера начать (для продолжения)
        max_examples: сколько всего примеров обработать (None = все)
    """
    import os
    
    # Создаём папку для выходных файлов
    os.makedirs(os.path.dirname(os.path.abspath(output_file)), exist_ok=True)
    
    # Определяем количество примеров
    total = len(dataset)
    if max_examples:
        total = min(total, max_examples)
    
    print(f"📊 Обработка {total} примеров по одному")
    print(f"   Максимальный лимит токенов: {max_new_tokens_limit}")
    print(f"   Коэффициент длины: {source_length_multiplier}")
    
    # Счётчики для статистики
    token_stats = {'min': float('inf'), 'max': 0, 'sum': 0, 'count': 0}
    length_stats = {'min': float('inf'), 'max': 0, 'sum': 0, 'count': 0}
    
    # Открываем файл для записи (в режиме append, если продолжаем)
    mode = 'a' if start_from > 0 else 'w'
    with open(output_file, mode, encoding='utf-8') as f:
        
        # Итерируем по одному примеру
        for idx in tqdm(range(start_from, total), desc="Перевод"):
            # Получаем данные
            source = dataset[idx]['translation']['en']
            reference = dataset[idx]['translation']['ru']
            
            # 🔢 Динамический расчет max_new_tokens
            estimated_source_tokens = len(source.split()) * 1.5
            current_max_new_tokens = int(estimated_source_tokens * source_length_multiplier)
            current_max_new_tokens = min(current_max_new_tokens, max_new_tokens_limit)
            
            # 🧮 Динамический расчет max_length для токенизатора
            current_max_length = int(current_max_new_tokens / source_length_multiplier) + 50
            
   
            # Формируем промпт
            prompt = system_prompt.format(text=source)
            
            # Токенизируем с динамическим max_length
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=current_max_length
            ).to(model.device)
            
            # Генерируем
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=current_max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            # Извлекаем только сгенерированную часть
            input_len = inputs.input_ids.shape[1]
            generated_tokens = outputs[0][input_len:]
            prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            
            # Записываем в файл
            record = {
                "id": idx,
                "source": source,
                "reference": reference,
                "translation": prediction.strip(),
                #"max_new_tokens_used": current_max_new_tokens,
                #"max_length_used": current_max_length,
                #"source_tokens_estimated": int(estimated_source_tokens)
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            #f.flush()


    
# === Пример использования ===
if __name__ == "__main__":
    from datasets import load_from_disk
    
    # Загрузка данных и модели
    dataset = load_from_disk("./wmt14_ru_en_validation_full")
    '''
    model_path = "tencent/HY-MT1.5-1.8B"  # или путь к локальной модели
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )'''
    
    # Системный промпт
    #SYSTEM_PROMPT = "请将以下英文翻译成俄语：{text}"
    SYSTEM_PROMPT = "把英语翻译成俄：{text}"
    
    
    from datetime import datetime

    print(datetime.now())
   
    # Запуск перевода
    #translate_dataset(
    translate_dataset_no_batch(
        dataset=dataset,
        tokenizer=tokenizer,
        model=model,
        system_prompt=SYSTEM_PROMPT,
        #batch_size=4,
        #max_new_tokens=200, # ?
        output_file="./outputs/translations_baseline.jsonl",
        max_examples=3000  # Для теста — 100 примеров
    )
    print(datetime.now())


2026-08-27 21:15:06.795125
📊 Обработка 3000 примеров по одному
   Максимальный лимит токенов: 2000
   Коэффициент длины: 2.2


Перевод: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [1:23:10<00:00,  1.66s/it]

2026-08-27 22:38:17.099008


In [1]:
SWITCH_LANG_TRHD = .8


import json
import re
from sacrebleu import corpus_bleu, CHRF
from tqdm import tqdm
from typing import List, Dict, Tuple

def is_russian(text: str, threshold: float = SWITCH_LANG_TRHD) -> bool:

    """
    Проверяет, является ли текст русским на основе доли кириллицы.
    """
    if not text or len(text.strip()) == 0:
        return True
    
    # Убираем пробелы, цифры, пунктуацию и эмодзи
    clean_text = re.sub(r'[\s\d\W]', '', text)
    if len(clean_text) == 0:
        return True
    
    cyrillic = len(re.findall(r'[а-яА-ЯёЁ]', clean_text))
    ratio = cyrillic / len(clean_text)
    return ratio >= threshold

def detect_language_switch(text: str) -> bool:
    """
    Возвращает True, если текст НЕ является русским (съезд на другой язык).
    """
    return not is_russian(text)

def compute_metrics_for_translations(filepath: str) -> Dict:
    """
    Загружает JSONL с переводами и вычисляет метрики.
    
    Returns:
        Словарь с метриками: средние значения по всем примерам
    """
    # Загружаем данные
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line))
    
    print(f"📂 Загружено {len(records)} записей из {filepath}")
    
    # Извлекаем тексты
    sources = [r['source'] for r in records]
    references = [r['reference'] for r in records]
    predictions = [r['translation'] for r in records]
    
    # 1. Считаем BLEU
    bleu_score = corpus_bleu(predictions, [references])
    print(f"📊 BLEU: {bleu_score.score:.2f}")
    
    # 2. Считаем chrF
    chrf_scorer = CHRF(word_order=2)
    chrf_score = chrf_scorer.corpus_score(predictions, [references])
    print(f"📊 chrF: {chrf_score.score:.2f}")
    
    # 3. Считаем "съезды" на другие языки
    switch_count = 0
    switch_examples = []
    
    for i, pred in enumerate(predictions):
        if detect_language_switch(pred):
            switch_count += 1
            if len(switch_examples) < 10:  # Сохраняем первые 10 примеров для анализа
                switch_examples.append({
                    'id': i,
                    'source': sources[i][:100] + "...",
                    'prediction': pred[:100] + "..."
                })
    
    switch_rate = switch_count / len(predictions) * 100
    print(f"📊 Съезды на другие языки: {switch_count} / {len(predictions)} ({switch_rate:.2f}%)")
    
    # 4. Статистика по длине
    src_lengths = [len(s.split()) for s in sources]
    ref_lengths = [len(r.split()) for r in references]
    pred_lengths = [len(p.split()) for p in predictions]
    
    print(f"\n📏 Статистика по длине (в словах):")
    print(f"  Источник: средняя {sum(src_lengths)/len(src_lengths):.1f}, "
          f"макс {max(src_lengths)}, мин {min(src_lengths)}")
    print(f"  Референс: средняя {sum(ref_lengths)/len(ref_lengths):.1f}, "
          f"макс {max(ref_lengths)}, мин {min(ref_lengths)}")
    print(f"  Перевод: средняя {sum(pred_lengths)/len(pred_lengths):.1f}, "
          f"макс {max(pred_lengths)}, мин {min(pred_lengths)}")
    
    # 5. Примеры съездов (для ручного анализа)
    if switch_examples:
        print(f"\n⚠️ Примеры съездов (первые {len(switch_examples)}):")
        for ex in switch_examples:
            print(f"  [{ex['id']}] Source: {ex['source']}")
            print(f"      Translation: {ex['prediction']}")
            print()
    
    # Возвращаем все метрики
    return {
        'total_examples': len(records),
        'bleu': bleu_score.score,
        'chrf': chrf_score.score,
        'switch_count': switch_count,
        'switch_rate': switch_rate,
        'avg_src_len': sum(src_lengths) / len(src_lengths),
        'avg_ref_len': sum(ref_lengths) / len(ref_lengths),
        'avg_pred_len': sum(pred_lengths) / len(pred_lengths),
        'switch_examples': switch_examples
    }

# Загружаем и анализируем переводы
metrics = compute_metrics_for_translations("./outputs/translations_baseline_3000.jsonl")

# Сохраняем метрики в отдельный файл
import json
with open("./outputs/metrics_baseline_3000.json", 'w', encoding='utf-8') as f:
    # Убираем примеры съездов, чтобы файл был компактным
    metrics_clean = {k: v for k, v in metrics.items() if k != 'switch_examples'}
    json.dump(metrics_clean, f, ensure_ascii=False, indent=2)

print("\n✅ Метрики сохранены в ./outputs/metrics_baseline.json")

📂 Загружено 3000 записей из ./outputs/translations_baseline_3000.jsonl
📊 BLEU: 11.50
📊 chrF: 35.25
📊 Съезды на другие языки: 981 / 3000 (32.70%)

📏 Статистика по длине (в словах):
  Источник: средняя 18.7, макс 82, мин 1
  Референс: средняя 16.2, макс 71, мин 1
  Перевод: средняя 24.3, макс 200, мин 0

⚠️ Примеры съездов (первые 10):
  [0] Source: A Republican strategy to counter the re-election of Obama...
      Translation: in 2016 is to try to make the election a two-way race.
Республиканская стратегия по противод...

  [6] Source: Unlike in Canada, the American States are responsible for the organisation of federal elections in t...
      Translation: В отличие от Канады, американские штаты несут ответственность за организацию федеральных выборов в С...

  [12] Source: Before the 2006 elections, no US State required voters to show a photo ID card....
      Translation: However, after the election, the US State of Texas decided to require all voters to present a photo ...

  [14] So